# Config

In [1]:
setup_conda_env = True
install_docking_tools = True


# Filepaths

In [2]:
conda_env_folder = "conda_envs"

mgltools_folder = "prep_tools/MGLToolsPckgs"
autodock_folder = "docking_tools/autodock"
autodock_gpu_folder = "docking_tools/autodock_gpu"
diffdock_folder = "docking_tools/diffdock"
equibind_folder = "docking_tools/equibind"

receptor_folder = "Data/Receptors"
ligand_folder = "Data/Ligands/JKU"    

# Imports

In [3]:
import sys
from pathlib import Path

# Setup Conda Environments

# Dock PoseBuster Benchmark Set with Autodock

In [ ]:
import yaml, json, time, threading, shutil
from pathlib import Path
from datetime import datetime

sys.path.insert(0, str(Path.cwd()))

from Scripts.Docking.run_autodock import (
    run_autodock_vina, build_prepared_manifest, generate_summary,
    print_summary, get_cpu_model, precompute_properties,
    collect_files, get_pdbqt_dir, DockingResult,
)
from Scripts.Utilities.prep_docking import run_workflow, _write_box_file

# ── Paths ────────────────────────────────────────────────────────────────────
benchmark_dir = Path("Data/PoseBuster Benchmark Set")
output_base   = Path("Dockings/Benchmark")
log_dir       = Path("Dockings/Logs/benchmark_logs")
config_path   = Path("Scripts/Docking/autodock_vina_docking_config.yaml")

# ── Load base config and override output paths ──────────────────────────────
with open(config_path) as f:
    base_cfg = yaml.safe_load(f)

base_cfg["output_dir"] = str(output_base)
base_cfg["log_dir"]    = str(log_dir)

output_base.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

cpu_model = get_cpu_model()
print(f"CPU: {cpu_model}")

# ── Determine converter(s) from config ───────────────────────────────────────
prep_tool = base_cfg.get("prep_tool", "mgltools")
converters: list[tuple[str, str]] = []
if prep_tool in ("mgltools", "both"):
    converters.append(("mgl_tools", "mgltools"))
if prep_tool in ("meeko", "both"):
    converters.append(("meeko", "meeko"))
print(f"Converters: {[c[0] for c in converters]}")


def box_from_sdf(sdf_path: Path, padding: float = 10.0):
    """Compute docking-box center & size from an SDF ligand (crystal pose)."""
    from rdkit import Chem
    suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=False)
    mol = next(iter(suppl))
    if mol is None:
        raise ValueError(f"Could not read molecule from {sdf_path}")
    conf = mol.GetConformer()
    xs, ys, zs = [], [], []
    for i in range(mol.GetNumAtoms()):
        pos = conf.GetAtomPosition(i)
        xs.append(pos.x); ys.append(pos.y); zs.append(pos.z)
    center = ((max(xs) + min(xs)) / 2, (max(ys) + min(ys)) / 2, (max(zs) + min(zs)) / 2)
    size   = ((max(xs) - min(xs)) + padding, (max(ys) - min(ys)) + padding, (max(zs) - min(zs)) + padding)
    return center, size


# ── Discover benchmark complexes ─────────────────────────────────────────────
complex_dirs = sorted(
    d for d in benchmark_dir.iterdir()
    if d.is_dir() and not d.name.startswith(("_", "."))
)
print(f"Found {len(complex_dirs)} benchmark complexes\n")

# ── Main loop ────────────────────────────────────────────────────────────────
all_results: list[DockingResult] = []
skipped, failed_prep = 0, 0

for idx, cdir in enumerate(complex_dirs, 1):
    pdb_id         = cdir.name                                     # e.g. "5S8I_2LY"
    protein_pdb    = cdir / f"{pdb_id}_protein.pdb"
    ligand_crystal = cdir / f"{pdb_id}_ligand.sdf"                 # crystal pose → defines box
    ligand_start   = cdir / f"{pdb_id}_ligand_start_conf.sdf"      # generated conf → dock this

    # ── Skip if files missing ────────────────────────────────────────────────
    if not protein_pdb.exists() or not ligand_start.exists() or not ligand_crystal.exists():
        print(f"[{idx}/{len(complex_dirs)}] SKIP {pdb_id} — missing files")
        skipped += 1
        continue

    # ── Skip if already docked (all converters) ──────────────────────────────
    done_markers = [output_base / pdb_id / cn / "docking_summary.json" for cn, _ in converters]
    if all(m.exists() for m in done_markers):
        print(f"[{idx}/{len(complex_dirs)}] ✓ {pdb_id} — already docked, skipping")
        skipped += 1
        continue

    print(f"\n{'─' * 60}")
    print(f"[{idx}/{len(complex_dirs)}] {pdb_id}")
    print(f"{'─' * 60}")

    # ── Create staging directories (one receptor, one ligand) ────────────────
    staging     = output_base / pdb_id / "_staging"
    rec_staging = staging / "receptors"
    lig_staging = staging / "ligands"
    rec_staging.mkdir(parents=True, exist_ok=True)
    lig_staging.mkdir(parents=True, exist_ok=True)

    # Symlink source files into staging (avoids copies)
    rec_link = rec_staging / protein_pdb.name
    lig_link = lig_staging / ligand_start.name
    if not rec_link.exists():
        rec_link.symlink_to(protein_pdb.resolve())
    if not lig_link.exists():
        lig_link.symlink_to(ligand_start.resolve())

    # ── Compute docking box from crystal ligand (10 Å padding) ───────────────
    try:
        center, size = box_from_sdf(ligand_crystal, padding=10.0)
    except Exception as exc:
        print(f"  ✗ Box computation failed: {exc}")
        failed_prep += 1
        continue

    # ── Iterate over converters ──────────────────────────────────────────────
    for conv_name, conv_arg in converters:
        vina_out = output_base / pdb_id / conv_name
        if (vina_out / "docking_summary.json").exists():
            print(f"  ✓ [{conv_name}] already docked — skipping")
            continue

        # ── Prepare protein PDBQT + box ──────────────────────────────────────
        protein_pdbqt_dir = get_pdbqt_dir(rec_staging)
        protein_outputs = run_workflow(
            input_dir=rec_staging,
            contains="proteins",
            output_dir=protein_pdbqt_dir,
            skip_pdb_validation=base_cfg.get("skip_pdb_validation", False),
            custom_postfix=f"_{conv_name}",
            process_postfixes=base_cfg.get("process_postfixes", False),
            repair_terminals=base_cfg.get("repair_terminals", False),
            converter=conv_arg,
            convert_proteins=True,
            verbose=False,
        )

        # Overwrite auto-generated box files with ligand-centered box
        for box_file in protein_pdbqt_dir.glob("*.box.txt"):
            _write_box_file(box_file, center, size)

        # ── Prepare ligand PDBQT (Meeko) ────────────────────────────────────
        ligand_pdbqt_dir = get_pdbqt_dir(lig_staging)
        ligand_outputs = run_workflow(
            input_dir=lig_staging,
            contains="ligands",
            output_dir=ligand_pdbqt_dir,
            process_postfixes=False,
            convert_ligands_with_meeko=True,
            verbose=False,
        )

        # ── Build manifest and dock ──────────────────────────────────────────
        manifest = build_prepared_manifest(protein_outputs, ligand_outputs)
        n_prot = len(manifest["proteins"])
        n_lig  = len(manifest["ligands"])
        if n_prot == 0 or n_lig == 0:
            print(f"  ✗ [{conv_name}] manifest empty (proteins={n_prot}, ligands={n_lig})")
            failed_prep += 1
            continue

        vina_out.mkdir(parents=True, exist_ok=True)

        results_df, results = run_autodock_vina(
            base_dir=vina_out,
            log_dir=log_dir,
            prepared_manifest=manifest,
            cfg=base_cfg,
            cpu_model=cpu_model,
            protein_workflow_data=protein_outputs,
            ligand_workflow_data=ligand_outputs,
        )

        # Save per-complex summary
        summary = generate_summary(results, base_cfg)
        with open(vina_out / "docking_summary.json", "w") as f:
            json.dump(summary, f, indent=2, default=str)

        for r in results:
            icon = "✓" if r.status == "success" else "✗"
            aff  = f"{r.best_affinity:.2f}" if r.best_affinity else "N/A"
            print(f"  {icon} [{conv_name}] {r.num_poses} poses | best: {aff} kcal/mol")
        all_results.extend(results)

# ── Final summary ────────────────────────────────────────────────────────────
n_ok   = sum(1 for r in all_results if r.status == "success")
n_fail = sum(1 for r in all_results if r.status == "failed")
print(f"\n{'=' * 60}")
print(f"BENCHMARK COMPLETE")
print(f"  Docked:       {n_ok}")
print(f"  Failed dock:  {n_fail}")
print(f"  Skipped:      {skipped}")
print(f"  Failed prep:  {failed_prep}")
print(f"  Results dir:  {output_base}")
print(f"{'=' * 60}")


CPU: Intel(R) Core(TM) i9-14900HX
Converters: ['mgl_tools', 'meeko']
Found 428 benchmark complexes


────────────────────────────────────────────────────────────
[1/428] 5S8I_2LY
────────────────────────────────────────────────────────────

PROCESSING 1 PROTEIN STRUCTURE(S)
Docking receptors: 1 | ligands: 1 | scoring: vina
Batch mode: OFF (per-ligand docking)
Docking log: /home/manndo/master_dev/Dockings/Benchmark/5S8I_2LY/mgl_tools/docking_log_mgl_tools.csv (0 existing entries)
Total docking jobs: 1  |  Workers: 1
  [1/1] ✓ 5S8I_2LY_protein_mgl_tools x 5S8I_2LY_ligand_start_conf -> success | 10 poses | best: -6.28 kcal/mol | 1.1s
  📝 Appended 1 entries to docking_log_mgl_tools.csv

Docking complete in 1.1s (0.0 min)
  Success: 1  |  Failed: 0  |  Skipped: 0
  ✓ [mgl_tools] 10 poses | best: -6.28 kcal/mol

PROCESSING 1 PROTEIN STRUCTURE(S)
Docking receptors: 1 | ligands: 1 | scoring: vina
Batch mode: OFF (per-ligand docking)
Docking log: /home/manndo/master_dev/Dockings/Benchmark/5S8I_

# Prepare and Create Orai Receptor Files (PDB)

In [4]:
sys.path.insert(0, str(Path.cwd().parents[1])) 

# Batch clean directory
from Scripts.Utilities.prepare_receptor_pdb import batch_clean_pdbs
batch_clean_pdbs(f"{receptor_folder}/Original", pattern='*.pdb', output_dir=receptor_folder)


Cleaning 4 PDB files from Data/Receptors/Original
Converting PDB: Orai1WT-MDSnap-Fr300.pdb → Orai1WT-MDSnap-Fr300.pdb
  Segment → Chain mapping: {'MONA': 'A', 'MONB': 'B', 'MONC': 'C', 'MOND': 'D', 'MONE': 'E', 'MONF': 'F'}
  Input atoms:      21026
  Output atoms:     10362
  Chains written:   6 (A, B, C, D, E, F)
  Hydrogens removed:10592
  Cap atoms removed:72
  Altloc dropped:   0
  HIS renamed:      600
  ✓ Saved: Data/Receptors/Orai1WT-MDSnap-Fr300.pdb
Converting PDB: Orai1WT-MDSnap-Fr400.pdb → Orai1WT-MDSnap-Fr400.pdb
  Segment → Chain mapping: {'MONA': 'A', 'MONB': 'B', 'MONC': 'C', 'MOND': 'D', 'MONE': 'E', 'MONF': 'F'}
  Input atoms:      21026
  Output atoms:     10362
  Chains written:   6 (A, B, C, D, E, F)
  Hydrogens removed:10592
  Cap atoms removed:72
  Altloc dropped:   0
  HIS renamed:      600
  ✓ Saved: Data/Receptors/Orai1WT-MDSnap-Fr400.pdb
Converting PDB: Orai1WT-START-Fr0.pdb → Orai1WT-START-Fr0.pdb
  Segment → Chain mapping: {'MONA': 'A', 'MONB': 'B', 'MONC':

['Data/Receptors/Orai1WT-MDSnap-Fr300.pdb',
 'Data/Receptors/Orai1WT-MDSnap-Fr400.pdb',
 'Data/Receptors/Orai1WT-START-Fr0.pdb',
 'Data/Receptors/Orai1WT-MDSnap-Fr499.pdb']

# Prepare Ligand Files (PDB)

In [5]:
from Scripts.Utilities.prep_docking import run_workflow

ligand_outputs = run_workflow(
    input_dir=Path(ligand_folder),
    contains="ligands",
    ligand_formats=["pdb"],
    output_dir=Path(ligand_folder),
    process_postfixes=False,
)



ORAI1 PROTEIN-LIGAND DOCKING PREPARATION WORKFLOW

STEP 1: CONVERTING LIGAND FILES (XYZ → PDB/SDF/MOL2)
✓ OpenBabel is installed

PROCESSING 3 LIGAND(S)
⚡ Processing 3 ligand XYZ files with 3 parallel workers

Converting: /home/manndo/master_dev/Data/Ligands/JKU/2abp-nh2-OPT.xyz

Converting: /home/manndo/master_dev/Data/Ligands/JKU/Synta-66-OPT-Singlet.xyz

Converting: /home/manndo/master_dev/Data/Ligands/JKU/gsk7975a-deprot-OPT.xyz
✓ PDB  (    6119 bytes) → Data/Ligands/JKU/gsk7975a-deprot-OPT.pdb
✓ PDB  (    5200 bytes) → Data/Ligands/JKU/2abp-nh2-OPT.pdb
✓ PDB  (    6728 bytes) → Data/Ligands/JKU/Synta-66-OPT-Singlet.pdb
• 3 ligand(s) already in SDF/MOL2/PDB format — using as-is


STEP 2: VALIDATING PROTEIN STRUCTURES
• No protein files supplied; skipping validation


STEP 3: PREPARING FOR MOLECULAR DOCKING


WORKFLOW COMPLETE

Next Steps:
1. Review converted ligand files in the output directory
3. For AutoDock Vina, convert to PDBQT using Meeko when ligands and proteins are availa

# Autodock Vina

## Autodock Config

In [6]:
import yaml

config_path = Path("Scripts/Docking/autodock_vina_docking_config.yaml")

# Load current config
with open(config_path, "r") as f:
    docking_cfg = yaml.safe_load(f)

# ── Edit any values below, then run this cell to save ────────────────────────

# Input paths (synced with notebook variables by default)
docking_cfg["receptors_dir"]       = receptor_folder          # "Data/Receptors"
docking_cfg["ligand_dirs"]         = [ligand_folder]           # ["Data/Ligands/JKU"]

# Preparation tools
docking_cfg["prep_tool"]           = "both"         # "mgltools", "meeko", or "both"
docking_cfg["skip_pdb_validation"] = False          # True=skip, False=validate (default: False)
docking_cfg["process_postfixes"]   = False         # True=add _prepared/_pdbqt postfixes, False=keep original names (default: False)
docking_cfg["repair_terminals"]    = False         # True=repair, False=do not repair (default: False)

# Output
docking_cfg["output_dir"]          = "Dockings/vina_results"
docking_cfg["log_dir"]             = "Dockings/Logs/vina_logs"

# Vina executable
docking_cfg["vina_bin"]            = "/home/manndo/AutoDock-Vina/build/linux/release/vina"

# Scoring function
docking_cfg["scoring_function"]    = "vina"         # "vina", "vinardo", or "ad4"
docking_cfg["autogrid_bin"]        = "/usr/local/bin/autogrid4"

# Batch mode
docking_cfg["batch_mode"]          = None            # None=auto, True=force, False=disable
docking_cfg["batch_size"]          = 10
docking_cfg["batch_timeout"]       = 900

# Docking parameters
docking_cfg["exhaustiveness"]      = 32
docking_cfg["num_modes"]           = 10
docking_cfg["energy_range"]        = 3
docking_cfg["seed"]                = 42
docking_cfg["timeout_per_complex"] = 600

# Parallelism
docking_cfg["max_workers"]         = 1
docking_cfg["cpus_per_worker"]     = 32

# Overwrite settings
docking_cfg["overwrite_existing"]  = True
docking_cfg["overwrite_poses"]     = True
docking_cfg["overwrite_error_log"] = True

# ── Save updated config ─────────────────────────────────────────────────────
with open(config_path, "w") as f:
    yaml.dump(docking_cfg, f, default_flow_style=False, sort_keys=False)

print(f"✓ Config saved to {config_path}")
print(yaml.dump(docking_cfg, default_flow_style=False, sort_keys=False))


✓ Config saved to Scripts/Docking/autodock_vina_docking_config.yaml
receptors_dir: Data/Receptors
ligand_dirs:
- Data/Ligands/JKU
prep_tool: both
skip_pdb_validation: false
process_postfixes: false
repair_terminals: false
output_dir: Dockings/vina_results
log_dir: Dockings/Logs/vina_logs
vina_bin: /home/manndo/AutoDock-Vina/build/linux/release/vina
scoring_function: vina
autogrid_bin: /usr/local/bin/autogrid4
batch_mode: null
batch_size: 10
batch_timeout: 900
exhaustiveness: 32
num_modes: 10
energy_range: 3
seed: 42
timeout_per_complex: 600
max_workers: 1
cpus_per_worker: 32
overwrite_existing: true
overwrite_poses: true
overwrite_error_log: true



In [7]:
%run Scripts/Docking/run_autodock.py -c Scripts/Docking/autodock_vina_docking_config.yaml

AutoDock Vina Batch Docking Configuration
Vina executable:     /home/manndo/AutoDock-Vina/build/linux/release/vina
Scoring function:    vina
Prep tool(s):        both
Batch mode:          auto
Batch size:          10
Batch timeout:       900s
Exhaustiveness:      32
Num modes:           10
Energy range:        3 kcal/mol
Timeout per complex: 600s (10 min)
Workers:             1  (CPUs/worker: 32)
Overwrite docking:   True
Overwrite poses:     True
Output dir:          Dockings/vina_results
Log dir:             Dockings/Logs/vina_logs
Receptor dir:        Data/Receptors
Ligand dirs (1):
  • Data/Ligands/JKU

CPU: Intel(R) Core(TM) i9-14900HX

────────────────────────────────────────────────────────────────────────────────
Preparing proteins with: mgl_tools
────────────────────────────────────────────────────────────────────────────────

ORAI1 PROTEIN-LIGAND DOCKING PREPARATION WORKFLOW

STEP 1: CONVERTING LIGAND FILES (XYZ → PDB/SDF/MOL2)
• No ligand files supplied; skipping conversion


In [ ]:
# Create a cell that dock each ligand from the posebusters benchmark set with all the receptors from the receptor_folder (not the subfolders)